# Laboratorio #1 — Entrenamiento de Redes Neuronales (MLP)
## CC3092 · Deep Learning y Sistemas Inteligentes

**Problema:** Regresión sobre el dataset *California Housing Prices*. El objetivo es predecir el valor mediano de una vivienda (`median_house_value`) a partir de características socioeconómicas y geográficas de distritos censales de California, usando un **Perceptrón Multicapa (MLP)** implementado en **PyTorch**.

**Repositorio:** _(colocar aquí el enlace al repositorio Git antes de la entrega)_

---

### Contenido del notebook
0. Configuración del entorno
1. Carga y exploración de los datos (§2)
2. Preparación de los datos: limpieza, codificación, escalado y *split* (§2)
3. Investigación: capas de `torch.nn` y optimizadores (§3)
4. Definición del modelo MLP y funciones de entrenamiento/evaluación (§4)
5. Entrenamiento e iteración de hiperparámetros — 10+ configuraciones (§4)
6. Selección del mejor modelo y evaluación única sobre *test* (§4)
7. Tabla resumen de iteraciones (§5)

## 0. Configuración del entorno

Importamos las librerías y fijamos una **semilla** para que los resultados sean reproducibles (mismo *split*, misma inicialización de pesos y mismo orden de *batches* en cada corrida). También detectamos el *device* disponible (CPU / GPU).

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --- Reproducibilidad ---
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch: {torch.__version__}")
print(f"Device : {device}")
print(f"Semilla: {SEED}")

---
## 1. Carga y exploración de los datos

Usamos la versión de **Kaggle** (`data/housing.csv`), que a diferencia de la de `sklearn` incluye la variable **categórica** `ocean_proximity` y filas con **valores nulos**, lo que nos permite demostrar codificación de categóricas y manejo de nulos.

In [ ]:
# Carga del dataset
DATA_PATH = "../data/housing.csv"
df = pd.read_csv(DATA_PATH)

print(f"Observaciones (filas): {df.shape[0]:,}")
print(f"Variables (columnas) : {df.shape[1]}")
df.head()

In [ ]:
# Tipos de dato e información general
df.info()

In [ ]:
# Estadísticos descriptivos de las variables numéricas
df.describe().T

In [ ]:
# Valores nulos y duplicados
print("Valores nulos por columna:")
print(df.isnull().sum())
print(f"\nFilas duplicadas: {df.duplicated().sum()}")

# Variable categórica
print("\nNiveles de 'ocean_proximity':")
print(df["ocean_proximity"].value_counts())

In [ ]:
# Distribución de todas las variables numéricas
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[num_cols].hist(bins=50, figsize=(14, 9))
plt.suptitle("Distribución de las variables numéricas", y=1.0, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Diagnóstico de censura (capping) en el target y en algunas features
cap_target = (df["median_house_value"] >= 500_001).sum()
cap_income = (df["median_income"] >= 15.0001).sum()
cap_age = (df["housing_median_age"] >= 52).sum()
print(f"median_house_value censurado en 500,001 : {cap_target:,} filas ({cap_target/len(df):.1%})")
print(f"median_income tope en 15.0001            : {cap_income:,} filas")
print(f"housing_median_age tope en 52            : {cap_age:,} filas")

# Asimetría (skew) de las features numéricas: valores altos => cola larga / outliers
skew = df[num_cols].drop(columns=["median_house_value"]).skew().sort_values(ascending=False)
print("\nAsimetría (skew) de las features numéricas:")
print(skew.round(2))

In [ ]:
# Correlación de las variables numéricas con el target
corr = df[num_cols].corr()["median_house_value"].drop("median_house_value").sort_values()
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

corr.plot(kind="barh", ax=ax[0], color="steelblue")
ax[0].set_title("Correlación de cada feature con median_house_value")
ax[0].axvline(0, color="k", lw=0.8)

sc = ax[1].scatter(df["longitude"], df["latitude"], c=df["median_house_value"],
                   cmap="viridis", s=8, alpha=0.4)
ax[1].set_title("Distribución geográfica del precio")
ax[1].set_xlabel("longitude"); ax[1].set_ylabel("latitude")
plt.colorbar(sc, ax=ax[1], label="median_house_value")
plt.tight_layout()
plt.show()

### Respuestas a las preguntas de exploración (P2)

**a) ¿Cuántas observaciones y cuántas variables tiene el dataset?**
El dataset tiene **20,640 observaciones** (distritos censales) y **10 variables**: 9 *features* y 1 variable objetivo.

**b) ¿Qué representa cada variable y cuál es el *target*?**

| Variable | Tipo | Descripción |
|---|---|---|
| `longitude`, `latitude` | numérica | Coordenadas geográficas del distrito. |
| `housing_median_age` | numérica | Antigüedad mediana de las viviendas (años). |
| `total_rooms` | numérica | Nº total de habitaciones en el distrito. |
| `total_bedrooms` | numérica | Nº total de dormitorios en el distrito. |
| `population` | numérica | Población del distrito. |
| `households` | numérica | Nº de hogares del distrito. |
| `median_income` | numérica | Ingreso mediano de los hogares (en decenas de miles de USD). |
| `ocean_proximity` | **categórica** | Cercanía al océano (`<1H OCEAN`, `INLAND`, `NEAR OCEAN`, `NEAR BAY`, `ISLAND`). |
| **`median_house_value`** | numérica | **TARGET**: valor mediano de la vivienda (USD). |

El EDA muestra que `median_income` es la feature con **mayor correlación positiva** con el precio (~0.69), y geográficamente los precios altos se concentran en la costa (Bahía de San Francisco y Los Ángeles).

**c) ¿Hay valores nulos, duplicados o atípicos (outliers)? ¿Cómo los trató?**
- **Nulos:** sí, solo en `total_bedrooms` → **207 filas** (~1% del dataset). *Estrategia:* imputar con la **mediana** calculada **únicamente en el conjunto de entrenamiento** (para no filtrar información de validación/test). Se hace en el Bloque 2.
- **Duplicados:** **0 filas** duplicadas, no se elimina nada.
- **Outliers / censura:** las features de conteo (`total_rooms`, `population`, `households`, `total_bedrooms`) están muy **sesgadas a la derecha** (skew de 3.4 a 4.9), con colas largas. Además hay **censura (capping)**: el target está topado en `500,001` (**965 filas ≈ 4.7%**), `median_income` en `15.0001` (49 filas) y `housing_median_age` en `52` (1,273 filas). *Estrategia:* **no se eliminan** los outliers (son datos reales y quitarlos sesgaría el modelo); en su lugar se controla su efecto con el **escalado estandarizado** (Bloque 2). Se documenta la censura del target porque limita el error mínimo alcanzable en ese rango.

**d) ¿Qué variables son numéricas y cuáles categóricas? ¿Cómo codificó las categóricas?**
- **Numéricas (8 features):** todas menos `ocean_proximity`.
- **Categórica (1):** `ocean_proximity`. Se codifica con **One-Hot Encoding** (`pd.get_dummies`), generando una columna binaria por nivel. Es nominal (sin orden), por lo que One-Hot es más apropiado que un *label encoding* ordinal. Con 5 niveles el aumento de dimensionalidad es mínimo.

**e) ¿Fue necesario normalizar o escalar las variables numéricas?**
**Sí, es imprescindible.** Las features están en escalas radicalmente distintas (`median_income` ∈ [0.5, 15] frente a `total_rooms` ∈ [2, 39,320]). Sin escalar, las features de mayor magnitud dominarían el gradiente y el entrenamiento del MLP sería inestable y lento. Se aplica **estandarización** (`StandardScaler`: media 0, desviación 1), **ajustada solo con el train** y aplicada a val/test.

---
## 2. Preparación de los datos

Regla clave (vista en clase): **el conjunto de *test* no debe influir en ninguna decisión**. Por eso el orden es:

1. **Primero** dividimos en `train` / `val` / `test` (**70 / 15 / 15**). Con 20,640 filas, val y test quedan con ~3,100 ejemplos cada uno — suficientes para una estimación estable de las métricas — mientras el train se lleva la mayor parte de los datos, siguiendo la guía de clase (train 70-80%, val/test 10-15%).
2. Todo lo que "aprende" de los datos (mediana para imputar, columnas del *one-hot*, media y desviación del escalado) se **ajusta únicamente con el `train`** y luego se **aplica** a `val` y `test`.

Además, estandarizamos también el **target**: los valores están en cientos de miles de USD, y estandarizarlos (media 0, desv. 1) estabiliza el entrenamiento. Guardamos el `scaler_y` para **invertir** las predicciones y reportar las métricas (MSE, MAE, RMSE) en **dólares reales**.

In [ ]:
# --- 1) Separar features (X) y target (y) ---
target_col = "median_house_value"
X = df.drop(columns=[target_col]).copy()
y = df[[target_col]].copy()

cat_cols = ["ocean_proximity"]
num_cols_feat = [c for c in X.columns if c not in cat_cols]

# --- 2) Split 70/15/15 (primero test 15%, luego val del 85% restante) ---
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15 / 0.85, random_state=SEED)  # 0.1765 * 0.85 = 0.15

for name, part in [("train", X_train), ("val", X_val), ("test", X_test)]:
    print(f"{name:5s}: {part.shape[0]:>6,} filas  ({part.shape[0]/len(X):.0%})")

In [ ]:
# --- 3) Imputación de nulos: mediana calculada SOLO con el train ---
X_train, X_val, X_test = X_train.copy(), X_val.copy(), X_test.copy()

median_bedrooms = X_train["total_bedrooms"].median()
X_train["total_bedrooms"] = X_train["total_bedrooms"].fillna(median_bedrooms)
X_val["total_bedrooms"]   = X_val["total_bedrooms"].fillna(median_bedrooms)
X_test["total_bedrooms"]  = X_test["total_bedrooms"].fillna(median_bedrooms)

print(f"Mediana de total_bedrooms (train): {median_bedrooms:.1f}")
print("Nulos restantes tras imputar:",
      int(pd.concat([X_train, X_val, X_test])["total_bedrooms"].isnull().sum()))

# --- 4) One-Hot Encoding de la categórica (columnas del train como referencia) ---
X_train = pd.get_dummies(X_train, columns=cat_cols)
X_val   = pd.get_dummies(X_val,   columns=cat_cols)
X_test  = pd.get_dummies(X_test,  columns=cat_cols)

# Alinear val/test a las columnas del train (mismo conjunto y orden de dummies)
X_val  = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("\nColumnas finales tras one-hot:")
print(list(X_train.columns))

In [ ]:
# --- 5) Escalado estandarizado (StandardScaler) ajustado SOLO con el train ---
# Solo escalamos las columnas numéricas; las dummies (0/1) se dejan intactas.
scaler_x = StandardScaler()
X_train[num_cols_feat] = scaler_x.fit_transform(X_train[num_cols_feat])
X_val[num_cols_feat]   = scaler_x.transform(X_val[num_cols_feat])
X_test[num_cols_feat]  = scaler_x.transform(X_test[num_cols_feat])

# Estandarizamos también el target (se invertirá para reportar métricas en USD)
scaler_y = StandardScaler()
y_train_s = scaler_y.fit_transform(y_train)
y_val_s   = scaler_y.transform(y_val)
y_test_s  = scaler_y.transform(y_test)

print("Media/desv del target (train):",
      f"mean={scaler_y.mean_[0]:,.0f}  std={scaler_y.scale_[0]:,.0f}")
print("Features numéricas escaladas (media≈0, std≈1):")
print(X_train[num_cols_feat].describe().loc[["mean", "std"]].T.round(2))

In [ ]:
# --- 6) Conversión a tensores de PyTorch ---
def to_tensor(arr):
    return torch.tensor(np.asarray(arr, dtype=np.float32))

X_train_t = to_tensor(X_train.astype(np.float32).values)
X_val_t   = to_tensor(X_val.astype(np.float32).values)
X_test_t  = to_tensor(X_test.astype(np.float32).values)
y_train_t = to_tensor(y_train_s)
y_val_t   = to_tensor(y_val_s)
y_test_t  = to_tensor(y_test_s)

n_features = X_train_t.shape[1]

# Datasets (los DataLoaders se crean por iteración, ya que batch_size es un hiperparámetro)
train_ds = TensorDataset(X_train_t, y_train_t)
val_ds   = TensorDataset(X_val_t,   y_val_t)
test_ds  = TensorDataset(X_test_t,  y_test_t)

print(f"n_features de entrada: {n_features}")
print(f"X_train: {tuple(X_train_t.shape)} | X_val: {tuple(X_val_t.shape)} | X_test: {tuple(X_test_t.shape)}")
print(f"y_train: {tuple(y_train_t.shape)} | y_val: {tuple(y_val_t.shape)} | y_test: {tuple(y_test_t.shape)}")

---
## 3. Investigación: capas de `torch.nn` y optimizadores

### 3.1 Capas para construir el MLP

| Capa | Propósito | Parámetros clave |
|---|---|---|
| **`nn.Linear`** | Capa densa: aplica `y = xWᵀ + b`. Es el bloque base del MLP; conecta todas las entradas con todas las neuronas de salida. | `in_features`, `out_features`, `bias` (usar sesgo, por defecto `True`). |
| **`nn.ReLU`** | Activación no lineal `max(0, x)`. Barata y sin saturación en positivos → converge rápido. Su riesgo son las *dying neurons* (neuronas que quedan en 0). | *(sin parámetros relevantes)* |
| **`nn.LeakyReLU`** | Variante de ReLU con pendiente pequeña para negativos (`x` si `x>0`, `αx` si `x≤0`). Evita las neuronas muertas. | `negative_slope` (α, típ. 0.01). |
| **`nn.Tanh`** | Activación `tanh(x)` con salida en (−1, 1), centrada en 0. Satura en los extremos → puede causar gradientes que se desvanecen en redes profundas. | *(sin parámetros relevantes)* |
| **`nn.Dropout`** | Regularización: durante el entrenamiento apaga aleatoriamente una fracción de neuronas, forzando redundancia y reduciendo *overfitting*. Se desactiva en evaluación. | `p` = probabilidad de apagado (0.2–0.5 típico). |
| **`nn.BatchNorm1d`** | Normaliza las activaciones de cada *batch* (media 0, var 1) y las reescala con parámetros aprendibles. Estabiliza y acelera el entrenamiento, permite `lr` mayores y aporta ligera regularización. | `num_features` (= nº de neuronas de la capa previa). |

### 3.2 Funciones de pérdida para regresión

| Pérdida | Fórmula (por muestra) | Comportamiento |
|---|---|---|
| **`nn.MSELoss`** | `(ŷ − y)²` | Penaliza fuerte los errores grandes (cuadrático) → muy **sensible a outliers**. Es la más usada en regresión; su raíz es el RMSE. |
| **`nn.L1Loss`** (MAE) | `|ŷ − y|` | Penalización lineal → **robusta a outliers**. Trata todos los errores por igual; su gradiente es constante. |
| **`nn.SmoothL1Loss`** (Huber) | cuadrática si `|error| < β`, lineal si es mayor | **Híbrido**: precisa como MSE cerca del cero y robusta como L1 en los errores grandes. Parámetro `beta` marca el umbral de cambio. |

### 3.3 Optimizadores (`torch.optim`)

Dos parámetros comunes a los tres:
- **`lr` (learning rate):** tamaño del paso al actualizar los pesos. **Muy alto** → el entrenamiento diverge u oscila; **muy bajo** → converge lento o se estanca. Es el hiperparámetro más crítico.
- **`weight_decay` (regularización L2):** penaliza pesos grandes sumando `λ·‖w‖²` a la pérdida. Reduce el *overfitting* al mantener los pesos pequeños.

| Optimizador | Cómo actualiza | Qué lo diferencia |
|---|---|---|
| **`SGD`** | Paso en dirección del gradiente, igual para todos los pesos. Suele usarse con `momentum` (acumula inercia). | El más simple y predecible, pero **sensible al `lr`** y lento sin *momentum*. Con buen ajuste puede generalizar muy bien. |
| **`Adam`** | `lr` **adaptativo por parámetro**, combinando momento de 1er orden (media del gradiente) y de 2º (varianza). | **Convergencia rápida y robusta** con poco ajuste. Suele ser la opción por defecto en MLPs. |
| **`RMSprop`** | Divide el `lr` por la media móvil de los gradientes al cuadrado (adapta por parámetro, sin el momento de 1er orden de Adam). | Bueno con objetivos **no estacionarios/ruidosos**; es, en esencia, Adam sin el término de momento. |

**Función en el entrenamiento del MLP:** el optimizador es quien traduce los gradientes calculados por *backpropagation* en actualizaciones concretas de los pesos. La elección afecta directamente la **velocidad de convergencia** y la **estabilidad**: los adaptativos (Adam, RMSprop) suelen converger antes con menos ajuste, mientras que SGD requiere más cuidado con el `lr` pero puede lograr mejor generalización. En las iteraciones del Bloque 5 comparamos los tres.